# 02 — SCD Type 2: Full History

Schema:

```text
changed key -> expire old current row + insert new current row
new key     -> insert new current row
unchanged   -> keep as-is
```

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("scd_type_2_history")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.version

'4.0.2'

## Step 1 — Current dimension

In [2]:
current_schema = StructType([
    StructField("user_id", StringType(), False),
    StructField("country", StringType(), False),
    StructField("segment", StringType(), False),
    StructField("valid_from_raw", StringType(), False),
    StructField("valid_to_raw", StringType(), True),
    StructField("is_current", BooleanType(), False),
])

current = (
    spark.createDataFrame(
        [
            ("u1", "SK", "retail", "2026-01-01", None, True),
            ("u2", "CZ", "retail", "2026-01-01", None, True),
        ],
        current_schema,
    )
    .withColumn("valid_from", F.to_date("valid_from_raw"))
    .withColumn("valid_to", F.to_date("valid_to_raw"))
    .drop("valid_from_raw", "valid_to_raw")
)

current.show(truncate=False)
current.printSchema()

+-------+-------+-------+----------+----------+--------+
|user_id|country|segment|is_current|valid_from|valid_to|
+-------+-------+-------+----------+----------+--------+
|u1     |SK     |retail |true      |2026-01-01|NULL    |
|u2     |CZ     |retail |true      |2026-01-01|NULL    |
+-------+-------+-------+----------+----------+--------+

root
 |-- user_id: string (nullable = false)
 |-- country: string (nullable = false)
 |-- segment: string (nullable = false)
 |-- is_current: boolean (nullable = false)
 |-- valid_from: date (nullable = true)
 |-- valid_to: date (nullable = true)



## Step 2 — Incoming changes

In [3]:
incoming_schema = StructType([
    StructField("user_id", StringType(), False),
    StructField("country", StringType(), False),
    StructField("segment", StringType(), False),
    StructField("change_date_raw", StringType(), False),
])

incoming = (
    spark.createDataFrame(
        [
            ("u1", "SK", "premium", "2026-02-01"),
            ("u2", "CZ", "retail", "2026-02-01"),
            ("u3", "DE", "retail", "2026-02-01"),
        ],
        incoming_schema,
    )
    .withColumn("change_date", F.to_date("change_date_raw"))
    .drop("change_date_raw")
)

incoming.show(truncate=False)

+-------+-------+-------+-----------+
|user_id|country|segment|change_date|
+-------+-------+-------+-----------+
|u1     |SK     |premium|2026-02-01 |
|u2     |CZ     |retail |2026-02-01 |
|u3     |DE     |retail |2026-02-01 |
+-------+-------+-------+-----------+



## Step 3 — Detect new, changed, unchanged

In [4]:
joined = (
    incoming.alias("i")
    .join(current.filter("is_current").alias("c"), on="user_id", how="left")
    .select(
        "user_id",
        F.col("i.country").alias("new_country"),
        F.col("i.segment").alias("new_segment"),
        F.col("i.change_date"),
        F.col("c.country").alias("old_country"),
        F.col("c.segment").alias("old_segment"),
        F.col("c.valid_from").alias("old_valid_from"),
    )
    .withColumn("is_new_key", F.col("old_country").isNull())
    .withColumn(
        "is_changed",
        (~F.col("is_new_key")) &
        ((F.col("new_country") != F.col("old_country")) | (F.col("new_segment") != F.col("old_segment")))
    )
    .withColumn("is_unchanged", (~F.col("is_new_key")) & (~F.col("is_changed")))
)

joined.show(truncate=False)

+-------+-----------+-----------+-----------+-----------+-----------+--------------+----------+----------+------------+
|user_id|new_country|new_segment|change_date|old_country|old_segment|old_valid_from|is_new_key|is_changed|is_unchanged|
+-------+-----------+-----------+-----------+-----------+-----------+--------------+----------+----------+------------+
|u1     |SK         |premium    |2026-02-01 |SK         |retail     |2026-01-01    |false     |true      |false       |
|u2     |CZ         |retail     |2026-02-01 |CZ         |retail     |2026-01-01    |false     |false     |true        |
|u3     |DE         |retail     |2026-02-01 |NULL       |NULL       |NULL          |true      |false     |false       |
+-------+-----------+-----------+-----------+-----------+-----------+--------------+----------+----------+------------+



## Step 4 — Expire changed rows

In [5]:
expired_rows = (
    joined
    .filter("is_changed")
    .select(
        "user_id",
        F.col("old_country").alias("country"),
        F.col("old_segment").alias("segment"),
        F.col("old_valid_from").alias("valid_from"),
        F.date_sub(F.col("change_date"), 1).alias("valid_to"),
        F.lit(False).alias("is_current"),
    )
)

expired_rows.show(truncate=False)

+-------+-------+-------+----------+----------+----------+
|user_id|country|segment|valid_from|valid_to  |is_current|
+-------+-------+-------+----------+----------+----------+
|u1     |SK     |retail |2026-01-01|2026-01-31|false     |
+-------+-------+-------+----------+----------+----------+



## Step 5 — Insert new current rows

In [6]:
new_current_rows = (
    joined
    .filter("is_changed OR is_new_key")
    .select(
        "user_id",
        F.col("new_country").alias("country"),
        F.col("new_segment").alias("segment"),
        F.col("change_date").alias("valid_from"),
        F.lit(None).cast(DateType()).alias("valid_to"),
        F.lit(True).alias("is_current"),
    )
)

new_current_rows.show(truncate=False)

+-------+-------+-------+----------+--------+----------+
|user_id|country|segment|valid_from|valid_to|is_current|
+-------+-------+-------+----------+--------+----------+
|u1     |SK     |premium|2026-02-01|NULL    |true      |
|u3     |DE     |retail |2026-02-01|NULL    |true      |
+-------+-------+-------+----------+--------+----------+



## Step 6 — Keep unaffected rows and build result

In [7]:
changed_keys = joined.filter("is_changed").select("user_id")

unaffected_rows = current.join(changed_keys, on="user_id", how="left_anti")

scd2_result = (
    unaffected_rows
    .unionByName(expired_rows)
    .unionByName(new_current_rows)
    .orderBy("user_id", "valid_from")
)

scd2_result.show(truncate=False)

+-------+-------+-------+----------+----------+----------+
|user_id|country|segment|is_current|valid_from|valid_to  |
+-------+-------+-------+----------+----------+----------+
|u1     |SK     |retail |false     |2026-01-01|2026-01-31|
|u1     |SK     |premium|true      |2026-02-01|NULL      |
|u2     |CZ     |retail |true      |2026-01-01|NULL      |
|u3     |DE     |retail |true      |2026-02-01|NULL      |
+-------+-------+-------+----------+----------+----------+



## Step 7 — Control totals

In [8]:
totals = spark.createDataFrame(
    [
        ("current_rows", current.count()),
        ("incoming_rows", incoming.count()),
        ("changed_keys", joined.filter("is_changed").count()),
        ("new_keys", joined.filter("is_new_key").count()),
        ("unchanged_keys", joined.filter("is_unchanged").count()),
        ("final_rows", scd2_result.count()),
    ],
    StructType([StructField("metric", StringType(), False), StructField("value", LongType(), False)])
)

totals.show(truncate=False)

+--------------+-----+
|metric        |value|
+--------------+-----+
|current_rows  |2    |
|incoming_rows |3    |
|changed_keys  |1    |
|new_keys      |1    |
|unchanged_keys|1    |
|final_rows    |4    |
+--------------+-----+



In [9]:
spark.stop()